## Выгрузка данных

In [2]:
YOUR_KEY_ID = None # id статического ключа
YOUR_SECRET_KEY = None # секрет статического ключа
BUCKET_NAME = None # название бакета (без s3://)
ENDPOINT_URL = None  # Стандартный эндпоинт
REGION = None 

In [22]:
import os
from io import BytesIO

import boto3
import numpy as np
import polars as pl
import pandas as pd

import tqdm
from botocore.client import Config
from datasets import load_dataset

In [3]:
session = boto3.session.Session()
s3_client = session.client(
    's3',
    endpoint_url=ENDPOINT_URL,
    aws_access_key_id=YOUR_KEY_ID,
    aws_secret_access_key=YOUR_SECRET_KEY,
    config=Config(region_name=REGION)
)

In [13]:
pools = ["train", "test", "val", "test_ranking", "val_ranking", "likes_train",
    "likes_val",
    "likes_test",
    "dislikes_train",
    "dislikes_val",
    "dislikes_test",
    "listens_train",
    "listens_val",
    "listens_test"]
s3_paths = {
    name: f"s3://{BUCKET_NAME}/data/yambda/{name}.parquet"
    for name in pools
}

In [7]:
scanning = lambda x: pl.scan_parquet(
    s3_paths[x],
    storage_options = {
        "aws_access_key_id": YOUR_KEY_ID,
        "aws_secret_access_key": YOUR_SECRET_KEY,
        "aws_region": "ru-central1-a",
        "endpoint_url": ENDPOINT_URL
    }
)

In [9]:
train = scanning("train")
test = scanning("test")
val = scanning("val")
test_ranking = scanning("test_ranking")
val_ranking = scanning("val_ranking")

In [15]:
likes_train = scanning("likes_train")
likes_val = scanning("likes_val")
likes_test = scanning("likes_test")

dislikes_train = scanning("dislikes_train")
dislikes_val = scanning("dislikes_val")
dislikes_test = scanning("dislikes_test")

listens_train = scanning("listens_train")
listens_val = scanning("listens_val")
listens_test = scanning("listens_test")

## Принесем статистику поближе

In [11]:
import polars as pl

for name, df in [
    ("train", train),
    ("val", val),
    ("test", test),
]:
    print(f"\n{name}")
    print(df.collect_schema())
    print(
        df.select(
            pl.len().alias("rows"),
            pl.col("uid").n_unique().alias("users"),
            pl.col("item_id").n_unique().alias("items"),
            pl.col("label").mean().alias("positive_rate"),
        ).collect()
    )


train
Schema([('uid', UInt32), ('item_id', UInt32), ('label', Int8), ('is_rec_skip', Boolean), ('is_dislike', Boolean)])
shape: (1, 4)
┌─────────┬───────┬────────┬───────────────┐
│ rows    ┆ users ┆ items  ┆ positive_rate │
│ ---     ┆ ---   ┆ ---    ┆ ---           │
│ u32     ┆ u32   ┆ u32    ┆ f64           │
╞═════════╪═══════╪════════╪═══════════════╡
│ 3161439 ┆ 8269  ┆ 330067 ┆ 0.264485      │
└─────────┴───────┴────────┴───────────────┘

val
Schema([('uid', UInt32), ('item_id', UInt32), ('label', Int8), ('is_rec_skip', Boolean), ('is_dislike', Boolean)])
shape: (1, 4)
┌───────┬───────┬───────┬───────────────┐
│ rows  ┆ users ┆ items ┆ positive_rate │
│ ---   ┆ ---   ┆ ---   ┆ ---           │
│ u32   ┆ u32   ┆ u32   ┆ f64           │
╞═══════╪═══════╪═══════╪═══════════════╡
│ 27123 ┆ 2805  ┆ 18084 ┆ 0.155329      │
└───────┴───────┴───────┴───────────────┘

test
Schema([('uid', UInt32), ('item_id', UInt32), ('label', Int8), ('is_rec_skip', Boolean), ('is_dislike', Boolean)])


In [16]:
source_tables = [
    "likes_train",
    "likes_val",
    "likes_test",
    "dislikes_train",
    "dislikes_val",
    "dislikes_test",
    "listens_train",
    "listens_val",
    "listens_test",
]

for name in source_tables:
    obj = globals().get(name)

    if isinstance(obj, pl.LazyFrame):
        print(f"\n{name}:")
        print(obj.collect_schema())
    else:
        print(f"{name}: not loaded")


likes_train:
Schema([('uid', UInt32), ('timestamp', UInt32), ('item_id', UInt32), ('is_organic', UInt8)])

likes_val:
Schema([('uid', UInt32), ('timestamp', UInt32), ('item_id', UInt32), ('is_organic', UInt8)])

likes_test:
Schema([('uid', UInt32), ('timestamp', UInt32), ('item_id', UInt32), ('is_organic', UInt8)])

dislikes_train:
Schema([('uid', UInt32), ('timestamp', UInt32), ('item_id', UInt32), ('is_organic', UInt8)])

dislikes_val:
Schema([('uid', UInt32), ('timestamp', UInt32), ('item_id', UInt32), ('is_organic', UInt8)])

dislikes_test:
Schema([('uid', UInt32), ('timestamp', UInt32), ('item_id', UInt32), ('is_organic', UInt8)])

listens_train:
Schema([('uid', UInt32), ('timestamp', UInt32), ('item_id', UInt32), ('is_organic', UInt8), ('played_ratio_pct', UInt16), ('track_length_seconds', UInt32)])

listens_val:
Schema([('uid', UInt32), ('timestamp', UInt32), ('item_id', UInt32), ('is_organic', UInt8), ('played_ratio_pct', UInt16), ('track_length_seconds', UInt32)])

listens_te

## Только простые user / item исторические агрегаты

!!! Это не бейзлайн, а проверка методологии. UID и ItemID потом исключим

Длительность можно брать из будущих событий только при условии, что это неизменяемое метаданное трека. Для строгости пока используем listens_train.

In [19]:
# Метаданные треков только из train-истории
item_metadata = (
    listens_train
    .group_by("item_id")
    .agg(
        pl.col("track_length_seconds")
        .median()
        .alias("track_length_seconds")
    )
)

FEATURES = [
    "uid",
    "item_id",
    "track_length_seconds",
]

CAT_FEATURES = ["uid", "item_id"]


def prepare_features(data: pl.LazyFrame) -> pl.LazyFrame:
    return (
        data
        .join(item_metadata, on="item_id", how="left")
        .with_columns(
            pl.col("uid").cast(pl.String),
            pl.col("item_id").cast(pl.String),
            pl.col("track_length_seconds")
              .cast(pl.Float32),
        )
        .select(
            "uid",
            "item_id",
            "label",
            "track_length_seconds",
        )
    )


train_features = prepare_features(train)
val_features = prepare_features(val)
test_features = prepare_features(test)

Проверяем пропуски

In [20]:
for name, data in [
    ("train", train_features),
    ("val", val_features),
    ("test", test_features),
]:
    print(name)

    print(
        data.select(
            pl.len().alias("rows"),
            pl.col("track_length_seconds")
              .null_count()
              .alias("missing_length"),
        ).collect()
    )

train
shape: (1, 2)
┌─────────┬────────────────┐
│ rows    ┆ missing_length │
│ ---     ┆ ---            │
│ u32     ┆ u32            │
╞═════════╪════════════════╡
│ 3161439 ┆ 11236          │
└─────────┴────────────────┘
val
shape: (1, 2)
┌───────┬────────────────┐
│ rows  ┆ missing_length │
│ ---   ┆ ---            │
│ u32   ┆ u32            │
╞═══════╪════════════════╡
│ 27123 ┆ 384            │
└───────┴────────────────┘
test
shape: (1, 2)
┌───────┬────────────────┐
│ rows  ┆ missing_length │
│ ---   ┆ ---            │
│ u32   ┆ u32            │
╞═══════╪════════════════╡
│ 28502 ┆ 485            │
└───────┴────────────────┘


### Обучение

In [ ]:
%pip install catboost

In [25]:
from catboost import CatBoostClassifier, Pool

In [26]:
train_df = train_features.collect().to_pandas()
val_df = val_features.collect().to_pandas()
test_df = test_features.collect().to_pandas()


for df in [train_df, val_df, test_df]:
    for col in CAT_FEATURES:
        df[col] = df[col].astype("category")


train_pool = Pool(
    data=train_df[FEATURES],
    label=train_df["label"],
    cat_features=CAT_FEATURES,
)

val_pool = Pool(
    data=val_df[FEATURES],
    label=val_df["label"],
    cat_features=CAT_FEATURES,
)

model_b0 = CatBoostClassifier(
    iterations=1000,
    depth=6,
    learning_rate=0.05,
    loss_function="Logloss",
    eval_metric="AUC",
    l2_leaf_reg=3,
    random_seed=42,
    thread_count=-1,
    verbose=100,
    allow_writing_files=False,
)

model_b0.fit(
    train_pool,
    eval_set=val_pool,
    early_stopping_rounds=100,
)

test_df["score"] = model_b0.predict_proba(
    test_df[FEATURES]
)[:, 1]

0:	test: 0.8380696	best: 0.8380696 (0)	total: 1.47s	remaining: 24m 32s
100:	test: 0.8438741	best: 0.8450430 (35)	total: 1m 31s	remaining: 13m 35s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.8450429772
bestIteration = 35

Shrink model to first 36 iterations.


### Метрики

In [27]:
from sklearn.metrics import roc_auc_score


def user_ndcg(y_true, y_score, k):
    order = np.argsort(-y_score)
    ranked = y_true[order][:k]

    discounts = 1.0 / np.log2(
        np.arange(2, len(ranked) + 2)
    )

    dcg = np.sum(ranked * discounts)

    n_positive = int(y_true.sum())
    ideal_len = min(n_positive, k)

    if ideal_len == 0:
        return np.nan

    ideal_discounts = 1.0 / np.log2(
        np.arange(2, ideal_len + 2)
    )

    idcg = ideal_discounts.sum()

    return dcg / idcg


def ranking_metrics(
    df,
    min_candidates=2,
):
    ndcg5 = []
    ndcg10 = []
    user_auc = []

    for uid, group in df.groupby(
        "uid", observed=True, sort=False
    ):
        if len(group) < min_candidates:
            continue

        y = group["label"].to_numpy()
        s = group["score"].to_numpy()

        if y.sum() == 0:
            continue

        ndcg5.append(user_ndcg(y, s, 5))
        ndcg10.append(user_ndcg(y, s, 10))

        if np.unique(y).size == 2:
            user_auc.append(roc_auc_score(y, s))

    return {
        "ndcg@5": float(np.mean(ndcg5))
                  if ndcg5 else np.nan,
        "ndcg@10": float(np.mean(ndcg10))
                   if ndcg10 else np.nan,
        "user_pairwise_auc": float(np.mean(user_auc))
                             if user_auc else np.nan,
        "n_users_ndcg": len(ndcg5),
        "n_users_auc": len(user_auc),
    }


def evaluate_model(df):
    result = {
        "roc_auc": roc_auc_score(
            df["label"],
            df["score"]
        )
    }

    ranking = ranking_metrics(
        df,
        min_candidates=2
    )

    ranking_10plus = ranking_metrics(
        df,
        min_candidates=10
    )

    result.update({
        "ndcg@5": ranking["ndcg@5"],
        "ndcg@10": ranking["ndcg@10"],
        "user_pairwise_auc":
            ranking["user_pairwise_auc"],
        "n_users_ranking":
            ranking["n_users_ndcg"],

        "ndcg@5_10plus":
            ranking_10plus["ndcg@5"],
        "ndcg@10_10plus":
            ranking_10plus["ndcg@10"],
        "user_pairwise_auc_10plus":
            ranking_10plus["user_pairwise_auc"],
        "n_users_10plus":
            ranking_10plus["n_users_ndcg"],
    })

    return result


results_b0 = evaluate_model(test_df)

pd.Series(results_b0).to_frame("CatBoost B0")

,CatBoost B0
roc_auc,0.844027
ndcg@5,0.641428
ndcg@10,0.688228
user_pairwise_auc,0.564357
n_users_ranking,971.000000
ndcg@5_10plus,0.388551
ndcg@10_10plus,0.461505
user_pairwise_auc_10plus,0.549383
n_users_10plus,409.000000


### Feature importance

In [28]:
importance_b0 = pd.DataFrame({
    "feature": FEATURES,
    "importance": model_b0.get_feature_importance(
        train_pool
    )
}).sort_values(
    "importance",
    ascending=False
)

importance_b0

,feature,importance
0,uid,93.377359
1,item_id,3.695146
2,track_length_seconds,2.927494


In [31]:
experiment_results = pd.DataFrame([
    {
        "experiment": "CatBoost B0",
        **results_b0,
    }
])

model_b0.save_model("catboost_b0.cbm")
experiment_results.to_csv(
    "results/baseline_b0.csv",
    index=False,
)

## Freeze

In [30]:
! pip freeze > requirements.txt

### Случайное ранжирование

In [34]:
def random_ranking_baseline(
    df,
    n_repeats=100,
    seed=42,
):
    rng = np.random.default_rng(seed)
    results = []

    groups = [
        g for _, g in df.groupby(
            "uid", observed=True, sort=False
        )
    ]

    for repeat in range(n_repeats):
        random_df = df.copy()

        random_df["score"] = rng.random(len(df))

        metrics = evaluate_model(random_df)
        results.append(metrics)

    return pd.DataFrame(results)


random_results = random_ranking_baseline(test_df)

comparison = pd.DataFrame({
    "CatBoost B0": pd.Series(results_b0),
    "Random mean": random_results.mean(),
    "Random std": random_results.std(),
})

comparison

,CatBoost B0,Random mean,Random std
roc_auc,0.844027,0.499950,0.004851
ndcg@5,0.641428,0.583738,0.005728
ndcg@10,0.688228,0.640096,0.005167
user_pairwise_auc,0.564357,0.499604,0.010971
n_users_ranking,971.000000,971.000000,0.000000
ndcg@5_10plus,0.388551,0.311277,0.010443
ndcg@10_10plus,0.461505,0.397535,0.008881
user_pairwise_auc_10plus,0.549383,0.500553,0.011921
n_users_10plus,409.000000,409.000000,0.000000


### Проверка роли пользователя

In [36]:
FEATURES_NO_USER = [
    "item_id",
    "track_length_seconds",
]

model_no_user = CatBoostClassifier(
    iterations=1000,
    depth=6,
    learning_rate=0.05,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=100,
    allow_writing_files=False,
)

model_no_user.fit(
    Pool(
        train_df[FEATURES_NO_USER],
        label=train_df["label"],
        cat_features=["item_id"],
    ),
    eval_set=Pool(
        val_df[FEATURES_NO_USER],
        label=val_df["label"],
        cat_features=["item_id"],
    ),
    early_stopping_rounds=100,
)

test_no_user = test_df.copy()

test_no_user["score"] = model_no_user.predict_proba(
    test_df[FEATURES_NO_USER]
)[:, 1]

results_no_user = evaluate_model(test_no_user)

pd.DataFrame({
    "CatBoost B0": results_b0,
    "Without user": results_no_user,
})

0:	test: 0.5541968	best: 0.5541968 (0)	total: 1.26s	remaining: 21m 2s
100:	test: 0.5720143	best: 0.5941708 (33)	total: 1m 25s	remaining: 12m 41s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.59417083
bestIteration = 33

Shrink model to first 34 iterations.


,CatBoost B0,Without user
roc_auc,0.844027,0.579092
ndcg@5,0.641428,0.605759
ndcg@10,0.688228,0.664041
user_pairwise_auc,0.564357,0.546419
n_users_ranking,971.000000,971.000000
ndcg@5_10plus,0.388551,0.335413
ndcg@10_10plus,0.461505,0.432162
user_pairwise_auc_10plus,0.549383,0.529714
n_users_10plus,409.000000,409.000000


## Ëпрст, придется восстанавливать ts

In [37]:
import polars as pl

KEYS = ["uid", "item_id"]

SPLITS = {
    "train": {
        "target": train,
        "likes": likes_train,
        "dislikes": dislikes_train,
        "listens": listens_train,
    },
    "val": {
        "target": val,
        "likes": likes_val,
        "dislikes": dislikes_val,
        "listens": listens_val,
    },
    "test": {
        "target": test,
        "likes": likes_test,
        "dislikes": dislikes_test,
        "listens": listens_test,
    },
}

In [38]:
def make_event_times(
    likes: pl.LazyFrame,
    dislikes: pl.LazyFrame,
    listens: pl.LazyFrame,
):
    like_times = (
        likes
        .group_by(KEYS)
        .agg(
            pl.col("timestamp")
            .min()
            .alias("like_ts")
        )
    )

    dislike_times = (
        dislikes
        .group_by(KEYS)
        .agg(
            pl.col("timestamp")
            .max()
            .alias("dislike_ts")
        )
    )

    skip_times = (
        listens
        .filter(
            (pl.col("is_organic") == 0)
            & (pl.col("played_ratio_pct") <= 50)
        )
        .group_by(KEYS)
        .agg(
            pl.col("timestamp")
            .min()
            .alias("skip_ts")
        )
    )

    return like_times, dislike_times, skip_times

In [39]:
def restore_timestamps(
    target: pl.LazyFrame,
    likes: pl.LazyFrame,
    dislikes: pl.LazyFrame,
    listens: pl.LazyFrame,
) -> pl.LazyFrame:

    like_ts, dislike_ts, skip_ts = make_event_times(
        likes, dislikes, listens
    )

    return (
        target
        .join(like_ts, on=KEYS, how="left")
        .join(dislike_ts, on=KEYS, how="left")
        .join(skip_ts, on=KEYS, how="left")
        .with_columns(
            pl.when(pl.col("label") == 1)
              .then(pl.lit("like"))
              .when(pl.col("is_dislike"))
              .then(pl.lit("dislike"))
              .when(pl.col("is_rec_skip"))
              .then(pl.lit("rec_skip"))
              .otherwise(pl.lit(None, dtype=pl.String))
              .alias("event_type"),

            pl.when(pl.col("label") == 1)
              .then(pl.col("like_ts"))
              .when(pl.col("is_dislike"))
              .then(pl.col("dislike_ts"))
              .when(pl.col("is_rec_skip"))
              .then(pl.col("skip_ts"))
              .otherwise(pl.lit(None, dtype=pl.UInt32))
              .alias("timestamp"),
        )
        .select(
            "uid",
            "item_id",
            "label",
            "timestamp",
            "event_type",
            "is_rec_skip",
            "is_dislike",
            "like_ts",
            "dislike_ts",
            "skip_ts",
        )
    )


train_v2 = restore_timestamps(**SPLITS["train"])
val_v2 = restore_timestamps(**SPLITS["val"])
test_v2 = restore_timestamps(**SPLITS["test"])

#### Небольшая проверка

In [41]:
def audit_restoration(name, old, new):

    stats = new.select(
        pl.len().alias("rows"),
        pl.col("timestamp")
          .null_count()
          .alias("missing_timestamp"),
        pl.col("timestamp")
          .min()
          .alias("min_timestamp"),
        pl.col("timestamp")
          .max()
          .alias("max_timestamp"),
        (
            (pl.col("is_rec_skip")) &
            (pl.col("is_dislike"))
        ).sum().alias("both_negative_sources"),
    ).collect()

    old_count = old.select(pl.len()).collect().item()
    new_count = stats["rows"][0]

    print(f"\n{name.upper()}")
    print(f"Old rows: {old_count:,}")
    print(f"New rows: {new_count:,}")
    print(stats)

    print(
        new.group_by("event_type")
           .agg(
               pl.len().alias("count"),
               pl.col("label").mean().alias("positive_rate"),
               pl.col("timestamp").null_count()
                 .alias("missing_timestamp"),
           )
           .collect()
           .sort("event_type")
    )


audit_restoration("train", train, train_v2)
audit_restoration("val", val, val_v2)
audit_restoration("test", test, test_v2)


TRAIN
Old rows: 3,161,439
New rows: 3,161,439
shape: (1, 5)
┌─────────┬───────────────────┬───────────────┬───────────────┬───────────────────────┐
│ rows    ┆ missing_timestamp ┆ min_timestamp ┆ max_timestamp ┆ both_negative_sources │
│ ---     ┆ ---               ┆ ---           ┆ ---           ┆ ---                   │
│ u32     ┆ u32               ┆ u32           ┆ u32           ┆ u32                   │
╞═════════╪═══════════════════╪═══════════════╪═══════════════╪═══════════════════════╡
│ 3161439 ┆ 0                 ┆ 10            ┆ 25823595      ┆ 35567                 │
└─────────┴───────────────────┴───────────────┴───────────────┴───────────────────────┘
shape: (3, 4)
┌────────────┬─────────┬───────────────┬───────────────────┐
│ event_type ┆ count   ┆ positive_rate ┆ missing_timestamp │
│ ---        ┆ ---     ┆ ---           ┆ ---               │
│ str        ┆ u32     ┆ f64           ┆ u32               │
╞════════════╪═════════╪═══════════════╪═══════════════════╡
│ di

In [42]:
for name, old, new in [
    ("train", train, train_v2),
    ("val", val, val_v2),
    ("test", test, test_v2),
]:
    columns = [
        "uid", "item_id", "label",
        "is_rec_skip", "is_dislike"
    ]

    missing = (
        old.select(columns)
        .join(
            new.select(columns),
            on=columns,
            how="anti",
        )
        .select(pl.len())
        .collect()
        .item()
    )

    print(f"{name}: rows not preserved = {missing}")

train: rows not preserved = 0
val: rows not preserved = 0
test: rows not preserved = 0


In [43]:
boundaries = []

for name, df in [
    ("train", train_v2),
    ("val", val_v2),
    ("test", test_v2),
]:
    row = df.select(
        pl.col("timestamp").min().alias("min_ts"),
        pl.col("timestamp").max().alias("max_ts"),
    ).collect().row(0, named=True)

    boundaries.append({
        "split": name,
        **row,
    })

pl.DataFrame(boundaries)

split,min_ts,max_ts
str,i64,i64
"""train""",10,25823595
"""val""",25825415,25911770
"""test""",25913600,26000000


Пупупу, придется еще проверить, насколько часто пользователь сначала пропускал трек, а затем дизлайкал — и наоборот.

In [45]:
def check_negative_timing(df):
    return (
        df.filter(
            pl.col("is_rec_skip") &
            pl.col("is_dislike")
        )
        .select(
            pl.len().alias("total"),

            (
                pl.col("skip_ts") <
                pl.col("dislike_ts")
            ).sum().alias("skip_before_dislike"),

            (
                pl.col("dislike_ts") <
                pl.col("skip_ts")
            ).sum().alias("dislike_before_skip"),

            (
                pl.col("skip_ts") ==
                pl.col("dislike_ts")
            ).sum().alias("same_time"),
        )
        .collect()
    )


for name, df in [
    ("train", train_v2),
    ("val", val_v2),
    ("test", test_v2),
]:
    print(f"\n{name}")
    print(check_negative_timing(df))


train
shape: (1, 4)
┌───────┬─────────────────────┬─────────────────────┬───────────┐
│ total ┆ skip_before_dislike ┆ dislike_before_skip ┆ same_time │
│ ---   ┆ ---                 ┆ ---                 ┆ ---       │
│ u32   ┆ u32                 ┆ u32                 ┆ u32       │
╞═══════╪═════════════════════╪═════════════════════╪═══════════╡
│ 35567 ┆ 27342               ┆ 657                 ┆ 7568      │
└───────┴─────────────────────┴─────────────────────┴───────────┘

val
shape: (1, 4)
┌───────┬─────────────────────┬─────────────────────┬───────────┐
│ total ┆ skip_before_dislike ┆ dislike_before_skip ┆ same_time │
│ ---   ┆ ---                 ┆ ---                 ┆ ---       │
│ u32   ┆ u32                 ┆ u32                 ┆ u32       │
╞═══════╪═════════════════════╪═════════════════════╪═══════════╡
│ 68    ┆ 48                  ┆ 3                   ┆ 17        │
└───────┴─────────────────────┴─────────────────────┴───────────┘

test
shape: (1, 4)
┌───────┬───────

Далеко эти штуки были друг от друга?

In [47]:
(
    train_v2
    .filter(
        pl.col("is_rec_skip") &
        pl.col("is_dislike")
    )
    .with_columns(
        (
            pl.col("dislike_ts").cast(pl.Int64)
            - pl.col("skip_ts").cast(pl.Int64)
        ).alias("time_delta")
    )
    .select(
        pl.col("time_delta").min().alias("min"),
        pl.col("time_delta").median().alias("median"),
        pl.col("time_delta").max().alias("max"),
    )
    .collect()
)

min,median,max
i64,f64,i64
-961280,185.0,25321335


target_event_ts — время целевого события, 

timestamp — точка отсечения исторических признаков. 

Пока они совпадают

In [48]:
def finalize_targets(df: pl.LazyFrame) -> pl.LazyFrame:
    return (
        df
        .rename({"timestamp": "target_event_ts"})
        .with_columns(
            pl.col("target_event_ts")
              .alias("timestamp")
        )
    )


train_v2 = finalize_targets(train_v2)
val_v2 = finalize_targets(val_v2)
test_v2 = finalize_targets(test_v2)

## Короче, переделываем экспы

In [69]:
import polars as pl
import numpy as np
import pandas as pd

from catboost import CatBoostClassifier, Pool
from sklearn.metrics import roc_auc_score

KEYS = ["uid", "item_id"]

def make_history_events(likes, dislikes, listens):

    l = (
        listens
        .select(
            "uid", "item_id", "timestamp",
            pl.lit(1, dtype=pl.Int32).alias("n_listens"),
            pl.lit(0, dtype=pl.Int32).alias("n_likes"),
            pl.lit(0, dtype=pl.Int32).alias("n_dislikes"),
            pl.col("played_ratio_pct")
              .cast(pl.Float64)
              .alias("play_sum"),
        )
    )

    pos = (
        likes
        .select(
            "uid", "item_id", "timestamp",
            pl.lit(0, dtype=pl.Int32).alias("n_listens"),
            pl.lit(1, dtype=pl.Int32).alias("n_likes"),
            pl.lit(0, dtype=pl.Int32).alias("n_dislikes"),
            pl.lit(0.0).alias("play_sum"),
        )
    )

    neg = (
        dislikes
        .select(
            "uid", "item_id", "timestamp",
            pl.lit(0, dtype=pl.Int32).alias("n_listens"),
            pl.lit(0, dtype=pl.Int32).alias("n_likes"),
            pl.lit(1, dtype=pl.Int32).alias("n_dislikes"),
            pl.lit(0.0).alias("play_sum"),
        )
    )

    return pl.concat(
        [l, pos, neg],
        how="vertical_relaxed",
    )


history = pl.concat(
    [
        make_history_events(
            likes_train, dislikes_train, listens_train
        ),
        make_history_events(
            likes_val, dislikes_val, listens_val
        ),
        make_history_events(
            likes_test, dislikes_test, listens_test
        ),
    ],
    how="vertical_relaxed",
)

Попросила группировать события с одинаковым timestamp до накопления статистик

In [70]:
COUNTERS = [
    "n_listens",
    "n_likes",
    "n_dislikes",
    "play_sum",
]


def make_cumulative_history(
    events: pl.LazyFrame,
    group_keys: list[str],
    prefix: str,
) -> pl.LazyFrame:

    grouped = (
        events
        .group_by(group_keys + ["timestamp"])
        .agg(
            [pl.col(c).sum().alias(c) for c in COUNTERS]
        )
        .sort(group_keys + ["timestamp"])
    )

    cumulative = grouped.with_columns(
        [
            pl.col(c)
              .cum_sum()
              .over(group_keys)
              .alias(f"{prefix}_{c}")
            for c in COUNTERS
        ]
    )


    return (
        cumulative
        .select(
            group_keys
            + ["timestamp"]
            + [f"{prefix}_{c}" for c in COUNTERS]
            + [
                pl.col("timestamp").alias(
                    f"{prefix}_last_ts"
                )
            ]
        )
        .sort("timestamp")
    )

In [90]:
### Собиоаем признаки

In [71]:
allow_exact_matches=False
def join_history(
    targets: pl.LazyFrame,
    stats: pl.LazyFrame,
    keys: list[str],
) -> pl.LazyFrame:

    return (
        targets
        .sort("timestamp")
        .join_asof(
            stats,
            on="timestamp",
            by=keys,
            strategy="backward",
            allow_exact_matches=False,
        )
    )

In [76]:
user_history = make_cumulative_history(
    history, ["uid"], "u"
)

item_history = make_cumulative_history(
    history, ["item_id"], "i"
)

ui_history = make_cumulative_history(
    history, ["uid", "item_id"], "ui"
)

# Начинаем с исходных целей, а не с частично
# сформированного features_all
features_all = targets_all

features_all = join_history(
    features_all,
    user_history,
    ["uid"],
)

features_all = join_history(
    features_all,
    item_history,
    ["item_id"],
)

features_all = join_history(
    features_all,
    ui_history,
    ["uid", "item_id"],
)

###### Дебаг

In [72]:
targets_all = pl.concat(
    [
        train_v2.with_columns(
            pl.lit("train").alias("split")
        ),
        val_v2.with_columns(
            pl.lit("val").alias("split")
        ),
        test_v2.with_columns(
            pl.lit("test").alias("split")
        ),
    ],
    how="vertical_relaxed",
).with_row_index("row_id")

In [77]:
print(features_all.collect_schema().names())

['row_id', 'uid', 'item_id', 'label', 'target_event_ts', 'event_type', 'is_rec_skip', 'is_dislike', 'like_ts', 'dislike_ts', 'skip_ts', 'timestamp', 'split', 'u_n_listens', 'u_n_likes', 'u_n_dislikes', 'u_play_sum', 'u_last_ts', 'i_n_listens', 'i_n_likes', 'i_n_dislikes', 'i_play_sum', 'i_last_ts', 'ui_n_listens', 'ui_n_likes', 'ui_n_dislikes', 'ui_play_sum', 'ui_last_ts']


In [78]:
assert not any(
    c.endswith("_right")
    for c in features_all.collect_schema().names()
)

###### Продолжаем

In [84]:
PREFIXES = ["u", "i", "ui"]

COUNTERS = [
    "n_listens",
    "n_likes",
    "n_dislikes",
    "play_sum",
]


def add_derived_features(df: pl.LazyFrame) -> pl.LazyFrame:
    expressions = []

    for p in PREFIXES:
        n_listens = pl.col(f"{p}_n_listens")
        n_likes = pl.col(f"{p}_n_likes")
        n_dislikes = pl.col(f"{p}_n_dislikes")
        play_sum = pl.col(f"{p}_play_sum")

        expressions.extend([
            # Сглаженная доля лайков
            (
                n_likes /
                (n_likes + n_dislikes + 1)
            ).alias(f"{p}_like_rate"),

            # Средняя глубина прослушивания
            pl.when(n_listens > 0)
              .then(play_sum / n_listens)
              .otherwise(None)
              .alias(f"{p}_avg_play_ratio"),

            # Время с момента последнего события
            (
                pl.col("timestamp").cast(pl.Int64)
                - pl.col(f"{p}_last_ts").cast(pl.Int64)
            ).alias(f"{p}_history_age"),
        ])

    return df.with_columns(expressions)


features_all = add_derived_features(features_all)

In [85]:
COUNT_FEATURES = [
    f"{p}_{c}"
    for p in PREFIXES
    for c in COUNTERS
]

features_all = features_all.with_columns([
    pl.col(c).fill_null(0)
    for c in COUNT_FEATURES
])

Предположим, что длительность трека не меняется и доступна из каталога на момент предсказания.

In [86]:
item_metadata = (
    listens_train
    .group_by("item_id")
    .agg(
        pl.col("track_length_seconds")
          .median()
          .cast(pl.Float32)
          .alias("track_length_seconds")
    )
)

features_all = features_all.join(
    item_metadata,
    on="item_id",
    how="left",
)

##### Проверочки

Попросила агента накидать функции верификации адекватности

In [87]:
print(features_all.collect_schema())

features_all.select([
    "uid",
    "item_id",
    "timestamp",
    "label",
    "u_n_listens",
    "u_like_rate",
    "i_n_listens",
    "i_like_rate",
    "ui_n_listens",
    "ui_avg_play_ratio",
    "track_length_seconds",
]).limit(10).collect()

Schema([('row_id', UInt32), ('uid', UInt32), ('item_id', UInt32), ('label', Int8), ('target_event_ts', UInt32), ('event_type', String), ('is_rec_skip', Boolean), ('is_dislike', Boolean), ('like_ts', UInt32), ('dislike_ts', UInt32), ('skip_ts', UInt32), ('timestamp', UInt32), ('split', String), ('u_n_listens', Int32), ('u_n_likes', Int32), ('u_n_dislikes', Int32), ('u_play_sum', Float64), ('u_last_ts', UInt32), ('i_n_listens', Int32), ('i_n_likes', Int32), ('i_n_dislikes', Int32), ('i_play_sum', Float64), ('i_last_ts', UInt32), ('ui_n_listens', Int32), ('ui_n_likes', Int32), ('ui_n_dislikes', Int32), ('ui_play_sum', Float64), ('ui_last_ts', UInt32), ('track_length_seconds', Float32), ('u_like_rate', Float64), ('u_avg_play_ratio', Float64), ('u_history_age', Int64), ('i_like_rate', Float64), ('i_avg_play_ratio', Float64), ('i_history_age', Int64), ('ui_like_rate', Float64), ('ui_avg_play_ratio', Float64), ('ui_history_age', Int64), ('track_length_seconds_right', Float32)])


sys:1: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
sys:1: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
sys:1: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided


uid,item_id,timestamp,label,u_n_listens,u_like_rate,i_n_listens,i_like_rate,ui_n_listens,ui_avg_play_ratio,track_length_seconds
u32,u32,u32,i8,i32,f64,i32,f64,i32,f64,f32
942900,6728180,10,0,0,0.0,0,0.0,0,null,270.0
794500,5559703,75,0,0,0.0,0,0.0,0,null,145.0
794500,6693536,75,0,0,0.0,0,0.0,0,null,245.0
794500,5890868,75,0,0,0.0,0,0.0,0,null,150.0
794500,1861694,75,0,0,0.0,0,0.0,0,null,245.0
794500,2224183,75,0,0,0.0,0,0.0,0,null,215.0
794500,7885371,75,0,0,0.0,0,0.0,0,null,95.0
794500,4955025,75,0,0,0.0,0,0.0,0,null,250.0
794500,2605934,75,0,0,0.0,0,0.0,0,null,180.0


In [88]:
audit = features_all.select(
    pl.len().alias("rows"),

    *[
        (
            pl.col(f"{p}_last_ts").is_not_null()
            & (
                pl.col(f"{p}_last_ts")
                >= pl.col("timestamp")
            )
        ).sum().alias(f"{p}_time_violations")
        for p in PREFIXES
    ],

    *[
        (pl.col(f"{p}_n_listens") < 0)
        .sum().alias(f"{p}_negative_counts")
        for p in PREFIXES
    ],
).collect()

print(audit)

sys:1: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
sys:1: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
sys:1: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided


shape: (1, 7)
┌─────────┬──────────────┬──────────────┬──────────────┬──────────────┬──────────────┬─────────────┐
│ rows    ┆ u_time_viola ┆ i_time_viola ┆ ui_time_viol ┆ u_negative_c ┆ i_negative_c ┆ ui_negative │
│ ---     ┆ tions        ┆ tions        ┆ ations       ┆ ounts        ┆ ounts        ┆ _counts     │
│ u32     ┆ ---          ┆ ---          ┆ ---          ┆ ---          ┆ ---          ┆ ---         │
│         ┆ u32          ┆ u32          ┆ u32          ┆ u32          ┆ u32          ┆ u32         │
╞═════════╪══════════════╪══════════════╪══════════════╪══════════════╪══════════════╪═════════════╡
│ 3217064 ┆ 0            ┆ 0            ┆ 0            ┆ 0            ┆ 0            ┆ 0           │
└─────────┴──────────────┴──────────────┴──────────────┴──────────────┴──────────────┴─────────────┘


In [89]:
pair_audit = features_all.select(
    (
        pl.col("ui_n_listens") > pl.col("u_n_listens")
    ).sum().alias("ui_gt_user"),

    (
        pl.col("ui_n_listens") > pl.col("i_n_listens")
    ).sum().alias("ui_gt_item"),
).collect()

print(pair_audit)

sys:1: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
sys:1: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
sys:1: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided


shape: (1, 2)
┌────────────┬────────────┐
│ ui_gt_user ┆ ui_gt_item │
│ ---        ┆ ---        │
│ u32        ┆ u32        │
╞════════════╪════════════╡
│ 0          ┆ 0          │
└────────────┴────────────┘


Судя по всему, по времени все ОК

### Сохраняем

In [91]:
features_all = features_all.sort("row_id")

features_all.sink_parquet(
    "yambda_historical_features_v1.parquet"
)

features_all = pl.scan_parquet(
    "yambda_historical_features_v1.parquet"
)

train_features_v2 = features_all.filter(
    pl.col("split") == "train"
)

val_features_v2 = features_all.filter(
    pl.col("split") == "val"
)

test_features_v2 = features_all.filter(
    pl.col("split") == "test"
)

/tmp/ipykernel_2880/2885339308.py:3: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  features_all.sink_parquet(
/tmp/ipykernel_2880/2885339308.py:3: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  features_all.sink_parquet(
/tmp/ipykernel_2880/2885339308.py:3: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  features_all.sink_parquet(


## Baseline

Наборы признаков

In [93]:
BASE_FEATURES = [
    "uid",
    "item_id",
    "track_length_seconds",
]

USER_FEATURES = [
    "u_n_listens",
    "u_n_likes",
    "u_n_dislikes",
    "u_like_rate",
    "u_avg_play_ratio",
    "u_history_age",
]

ITEM_FEATURES = [
    "i_n_listens",
    "i_n_likes",
    "i_n_dislikes",
    "i_like_rate",
    "i_avg_play_ratio",
    "i_history_age",
]

UI_FEATURES = [
    "ui_n_listens",
    "ui_n_likes",
    "ui_n_dislikes",
    "ui_avg_play_ratio",
    "ui_history_age",
]

FEATURE_SETS = {
    "B0": BASE_FEATURES,
    "B1a": BASE_FEATURES + USER_FEATURES,
    "B1b": BASE_FEATURES + USER_FEATURES + ITEM_FEATURES,
    "B1c": (
        BASE_FEATURES
        + USER_FEATURES
        + ITEM_FEATURES
        + UI_FEATURES
    ),
}

for name, features in FEATURE_SETS.items():
    print(f"{name}: {len(features)} features")

B0: 3 features
B1a: 9 features
B1b: 15 features
B1c: 20 features


In [94]:
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import roc_auc_score

In [95]:
ALL_FEATURES = FEATURE_SETS["B1c"]

CAT_FEATURES = ["uid", "item_id"]

COLUMNS = ["label"] + ALL_FEATURES


def prepare_catboost_data(lazy_df):
    df = (
        lazy_df
        .select(COLUMNS)
        .collect()
        .to_pandas()
    )

    for col in CAT_FEATURES:
        df[col] = df[col].astype(str)

    for col in ALL_FEATURES:
        if col not in CAT_FEATURES:
            df[col] = (
                pd.to_numeric(df[col], errors="coerce")
                .replace([np.inf, -np.inf], np.nan)
                .astype("float32")
            )

    df["label"] = df["label"].astype("int8")

    return df


train_cb = prepare_catboost_data(train_features_v2)
val_cb = prepare_catboost_data(val_features_v2)

print("Train:", train_cb.shape)
print("Val:", val_cb.shape)

Train: (3161439, 21)
Val: (27123, 21)


In [96]:
from pathlib import Path

Path("models").mkdir(exist_ok=True)

models = {}
val_predictions = {}
experiment_info = []


for name, features in FEATURE_SETS.items():

    print(f"\n{'=' * 20} {name} {'=' * 20}")

    cats = [
        c for c in features
        if c in CAT_FEATURES
    ]

    train_pool = Pool(
        data=train_cb[features],
        label=train_cb["label"],
        cat_features=cats,
    )

    val_pool = Pool(
        data=val_cb[features],
        label=val_cb["label"],
        cat_features=cats,
    )

    model = CatBoostClassifier(
        iterations=1500,
        depth=6,
        learning_rate=0.05,
        l2_leaf_reg=3,
        loss_function="Logloss",
        eval_metric="Logloss",
        random_seed=42,
        thread_count=-1,
        allow_writing_files=False,
        verbose=200,
    )

    model.fit(
        train_pool,
        eval_set=val_pool,
        early_stopping_rounds=100,
    )

    val_predictions[name] = model.predict_proba(
        val_cb[features]
    )[:, 1]

    models[name] = model

    model.save_model(
        f"models/catboost_{name.lower()}.cbm"
    )

    experiment_info.append({
        "model": name,
        "n_features": len(features),
        "best_iteration": model.best_iteration_,
        "best_val_logloss": model.best_score_[
            "validation"
        ]["Logloss"],
    })


==================== B0 ====================
0:	learn: 0.6590606	test: 0.6528809	best: 0.6528809 (0)	total: 1.36s	remaining: 33m 57s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.3367326066
bestIteration = 83

Shrink model to first 84 iterations.

==================== B1a ====================
0:	learn: 0.6545288	test: 0.6495171	best: 0.6495171 (0)	total: 1.94s	remaining: 48m 24s
200:	learn: 0.3520537	test: 0.3028990	best: 0.3028022 (196)	total: 5m 27s	remaining: 35m 14s
400:	learn: 0.3454711	test: 0.2996196	best: 0.2996196 (400)	total: 11m 19s	remaining: 31m 1s
600:	learn: 0.3419076	test: 0.2978884	best: 0.2978016 (599)	total: 17m 27s	remaining: 26m 6s
800:	learn: 0.3393834	test: 0.2966951	best: 0.2966451 (785)	total: 23m 31s	remaining: 20m 31s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.2963332448
bestIteration = 886

Shrink model to first 887 iterations.

==================== B1b ====================
0:	learn: 0.6531855	test: 0.6483374

Хм, что-то не то с B1c

## Метрики Baseline

Это чисто для меня, чтобы сравнить с прошлым хкспом (потом не буду включать в сравнение)

In [97]:
results = []

for name, scores in val_predictions.items():

    evaluated = val_cb[["uid", "label"]].copy()
    evaluated["score"] = scores

    metrics = evaluate_model(evaluated)

    results.append({
        "model": name,
        **metrics,
    })


metrics_df = (
    pd.DataFrame(results)
    .set_index("model")
)

metrics_df.T

model,B0,B1a,B1b,B1c
roc_auc,0.844212,0.887208,0.914981,0.752018
ndcg@5,0.631538,0.724928,0.802878,0.735052
ndcg@10,0.676314,0.765242,0.827908,0.773045
user_pairwise_auc,0.551748,0.680879,0.758061,0.698127
n_users_ranking,963.000000,963.000000,963.000000,963.000000
ndcg@5_10plus,0.360546,0.536491,0.684171,0.569707
ndcg@10_10plus,0.433645,0.609035,0.725143,0.637097
user_pairwise_auc_10plus,0.537563,0.718155,0.803082,0.735069
n_users_10plus,380.000000,380.000000,380.000000,380.000000


###### Основные метрики

In [98]:
main_metrics = [
    "ndcg@5_10plus",
    "ndcg@10_10plus",
    "user_pairwise_auc_10plus",
    "roc_auc",
]

comparison = metrics_df[main_metrics].copy()

comparison["ndcg5_gain_vs_B0"] = (
    comparison["ndcg@5_10plus"]
    - comparison.loc["B0", "ndcg@5_10plus"]
)

comparison["ndcg5_gain_pct"] = (
    100
    * comparison["ndcg5_gain_vs_B0"]
    / comparison.loc["B0", "ndcg@5_10plus"]
)

comparison.round(4)

,ndcg@5_10plus,ndcg@10_10plus,user_pairwise_auc_10plus,roc_auc,ndcg5_gain_vs_B0,ndcg5_gain_pct
model,,,,,,
B0,0.3605,0.4336,0.5376,0.8442,0.0000,0.0000
B1a,0.5365,0.6090,0.7182,0.8872,0.1759,48.7998
B1b,0.6842,0.7251,0.8031,0.9150,0.3236,89.7598
B1c,0.5697,0.6371,0.7351,0.7520,0.2092,58.0123


Главный показатель: ndcg@5_10plus. Именно по нему планирую в первую очередь смотреть, улучшилось ли персональное (попоолльзовательское) ранжирование.

##### Feature importance для B1c 

После добавления истории

In [101]:
importance_b1c = (
    pd.DataFrame({
        "feature": FEATURE_SETS["B1c"],
        "importance": models["B1c"].get_feature_importance(),
    })
    .sort_values(
        "importance",
        ascending=False,
    )
)

importance_b1c

,feature,importance
8,u_history_age,36.957905
0,uid,26.724991
18,ui_avg_play_ratio,24.106392
19,ui_history_age,7.506329
15,ui_n_listens,3.756162
3,u_n_listens,0.948221
2,track_length_seconds,0.000000
1,item_id,0.000000
7,u_avg_play_ratio,0.000000
4,u_n_likes,0.000000


In [102]:
def feature_group(feature):
    if feature in BASE_FEATURES:
        return "B0: Base"
    if feature in USER_FEATURES:
        return "B1a: User"
    if feature in ITEM_FEATURES:
        return "B1b: Item"
    if feature in UI_FEATURES:
        return "B1c: User-Item"
    return "Other"


group_importance = (
    importance_b1c
    .assign(
        group=lambda df: df["feature"].map(feature_group)
    )
    .groupby("group")["importance"]
    .sum()
    .sort_values(ascending=False)
)

group_importance

group
B1a: User         37.906125
B1c: User-Item    35.368884
B0: Base          26.724991
B1b: Item          0.000000
Name: importance, dtype: float64

In [103]:
experiment_info_df = pd.DataFrame(experiment_info)

experiment_summary = (
    experiment_info_df
    .merge(
        metrics_df.reset_index(),
        on="model",
    )
)

experiment_summary.to_csv(
    "results/catboost_ablation_v1.csv",
    index=False,
)

experiment_summary

,model,n_features,best_iteration,best_val_logloss,roc_auc,ndcg@5,ndcg@10,user_pairwise_auc,n_users_ranking,ndcg@5_10plus,ndcg@10_10plus,user_pairwise_auc_10plus,n_users_10plus
0,B0,3,83,0.336733,0.844212,0.631538,0.676314,0.551748,963,0.360546,0.433645,0.537563,380
1,B1a,9,886,0.296333,0.887208,0.724928,0.765242,0.680879,963,0.536491,0.609035,0.718155,380
2,B1b,15,527,0.259956,0.914981,0.802878,0.827908,0.758061,963,0.684171,0.725143,0.803082,380
3,B1c,20,5,0.633489,0.752018,0.735052,0.773045,0.698127,963,0.569707,0.637097,0.735069,380


!!!!! поскольку исходный B0 уже оценивался на test, и по нему я приняла решение о смене процедуры, теперь все решения о признаках принимать буду только по validation

In [ ]:
Разбираемся

In [104]:
ui_columns = [
    "ui_n_listens",
    "ui_n_likes",
    "ui_n_dislikes",
    "ui_avg_play_ratio",
    "ui_history_age",
]

for name, df in [
    ("train", train_df_v2 if "train_df_v2" in globals() else train_cb),
    ("val", val_df_v2 if "val_df_v2" in globals() else val_cb),
]:
    print(f"\n{name}")
    print(
        df[ui_columns]
        .describe(percentiles=[0.5, 0.9, 0.99])
        .T
    )


train
                       count          mean           std  min     50%  \
ui_n_listens       3161439.0  4.158590e-01  2.322788e+00  0.0     0.0   
ui_n_likes         3161439.0  0.000000e+00  0.000000e+00  0.0     0.0   
ui_n_dislikes      3161439.0  7.047424e-04  3.190654e-02  0.0     0.0   
ui_avg_play_ratio   535841.0  5.357292e+01  3.997983e+01  0.0    53.0   
ui_history_age      536214.0  1.088694e+06  2.900195e+06  5.0  1535.0   

                         90%         99%         max  
ui_n_listens             1.0         6.0       863.0  
ui_n_likes               0.0         0.0         0.0  
ui_n_dislikes            0.0         0.0        13.0  
ui_avg_play_ratio      100.0       100.0       159.0  
ui_history_age     3316896.5  15620059.0  25729216.0  

val
                     count          mean           std  min           50%  \
ui_n_listens       27123.0  3.302843e+00  7.121390e+00  0.0  0.000000e+00   
ui_n_likes         27123.0  0.000000e+00  0.000000e+00  0.0  0.00

In [106]:
for name, df in [
    ("train", train_features_v2),
    ("val", val_features_v2),
]:
    print(f"\n{name}")

    print(
        df.select(
            (pl.col("ui_n_listens") > 0)
                .mean()
                .alias("has_ui_history"),

            (pl.col("ui_n_likes") > 0)
                .mean()
                .alias("has_previous_like"),

            (pl.col("ui_n_dislikes") > 0)
                .mean()
                .alias("has_previous_dislike"),
        ).collect()
    )


train
shape: (1, 3)
┌────────────────┬───────────────────┬──────────────────────┐
│ has_ui_history ┆ has_previous_like ┆ has_previous_dislike │
│ ---            ┆ ---               ┆ ---                  │
│ f64            ┆ f64               ┆ f64                  │
╞════════════════╪═══════════════════╪══════════════════════╡
│ 0.169493       ┆ 0.0               ┆ 0.000618             │
└────────────────┴───────────────────┴──────────────────────┘

val
shape: (1, 3)
┌────────────────┬───────────────────┬──────────────────────┐
│ has_ui_history ┆ has_previous_like ┆ has_previous_dislike │
│ ---            ┆ ---               ┆ ---                  │
│ f64            ┆ f64               ┆ f64                  │
╞════════════════╪═══════════════════╪══════════════════════╡
│ 0.493382       ┆ 0.0               ┆ 0.000221             │
└────────────────┴───────────────────┴──────────────────────┘


In [108]:
! pip freeze > requirements_02.txt